# Drone Araç Tespiti: RF-DETR-L (tile + zoom) + TTA + ConvNeXt-T sınıflandırıcı

**Final submission (Public 0.80740):** tüm veriyle eğitilmiş RF-DETR-Large (7 epoch eğitim, **6. epoch EMA** ağırlıkları)
→ 3 geçişli TTA (zoom 1.33, zoom 1.33 + yatay çevirme, zoom 1.6) → sınıf bazlı NMS
→ ConvNeXt-Tiny kutu sınıflandırıcısıyla yeniden skorlama: `score^(1-α) · p[sınıf]^α`, α = 0.12.

| Adım | Val mAP@0.5 (sahne bazlı val) | Public |
|---|---|---|
| İlk model: rastgele split, zoom yok, 12 epoch | 0.843 (sızıntılı val) / 0.819 (sahne bazlı val) | 0.767 |
| Run B: sahne bazlı split + zoom 1.33 + augmentation, 6 epoch | 0.8384 | 0.78883 |
| + 3 geçişli TTA | 0.8453 | 0.79966 |
| + ConvNeXt-T sınıflandırıcı (α=0.12) | 0.8541 | 0.80358 |
| + tüm veriyle eğitim (val dahil), 6. epoch EMA | ölçülemez (val eğitimde) | **0.80740** |

Bölümler:
1. Kurulum
2. Sahne bazlı train/val ayrımı (`make_group_split.py`)
3. Tile + zoom veri seti (`preprocess.py`)
4. Ortak araçlar: tile'lı çıkarım, birleştirme, metrik, submission yazıcı
5. **Doğrulama deneyi** (Run B): eğitim → val skoru → TTA seçimi
6. **Sınıflandırıcı** (ConvNeXt-Tiny): crop veri seti → eğitim → α taraması
7. **Final**: tüm veriyle eğitim → 6. epoch EMA → test TTA + sınıflandırıcı → `submission.csv`

Donanım: 1× NVIDIA RTX PRO 6000 (96 GB), Colab. Bölüm 5–6 yalnızca ayar seçimi için gerekli. Final submission için 1–4 ve 6'daki sınıflandırıcı eğitimi, ardından 7 yeterli.

## 1. Kurulum

In [ ]:
# Kaggle API anahtarı: ~/.kaggle/kaggle.json ya da KAGGLE_API_TOKEN ortam değişkeni (anahtarı koda yazmayın)
!pip install -q -U kaggle "rfdetr[train]" timm transformers
!mkdir -p /content/data && cd /content/data && kaggle competitions download -q -c level-up-ai-roketsan-yapay-zeka-hackathonu && unzip -q -o '*.zip' && rm -f *.zip
import os
DATA, WORK = '/content/data', '/content/work'
OUT_DIR = '/content/drive/MyDrive/roketsan_hackathon'   # checkpoint'ler için kalıcı klasör (Drive)
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

## 2. Sahne bazlı train/val ayrımı
İlk rastgele split'te val skoru 0.843 çıkarken public 0.767 geldi. Sebep, aynı video/sahneden gelen karelerin hem train'e hem val'e düşmesiydi.
Çözüm şöyle: DINOv2-small embedding'leri (CLS + ortalama patch) hesaplanıyor, cosine benzerliği τ'dan büyük olan görüntüler union-find ile aynı gruba bağlanıyor.
τ, val→train en yakın komşu benzerlik dağılımı test→train dağılımına en yakın olacak şekilde seçiliyor (KS istatistiği). Seçilen değer τ = 0.9.
Val, testteki çözünürlük dağılımına göre grup grup dolduruluyor. 2000×1500 çözünürlük testte olmadığı için val'e hiç alınmıyor.
Sonuç: 5251 train / 1220 val, 3289 grup.

In [ ]:
%%writefile /content/make_group_split.py
"""Scene-grouped train/val split calibrated against the test set."""
import argparse, os, random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

p = argparse.ArgumentParser()
p.add_argument("--root", default=".")
p.add_argument("--out", default="work/split")
p.add_argument("--model", default="facebook/dinov2-small")
p.add_argument("--val_frac", type=float, default=0.18)
p.add_argument("--seed", type=int, default=42)
args = p.parse_args()
os.makedirs(args.out, exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"

tr_ids = sorted(f[:-4] for f in os.listdir(f"{args.root}/train/images"))
te_ids = sorted(f[:-4] for f in os.listdir(f"{args.root}/test/images"))
paths = [f"{args.root}/train/images/{i}.jpg" for i in tr_ids] + [f"{args.root}/test/images/{i}.jpg" for i in te_ids]
size = {pth: Image.open(pth).size for pth in paths}

# ---- embeddings (whole image, no center crop), cached
cache = f"{args.out}/emb_{args.model.split('/')[-1]}.npy"
if os.path.exists(cache):
    E = np.load(cache)
else:
    from transformers import AutoModel
    model = AutoModel.from_pretrained(args.model).to(dev).eval()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    E = []
    with torch.no_grad():
        for k in range(0, len(paths), 64):
            x = torch.stack([torch.from_numpy(np.asarray(Image.open(q).convert("RGB").resize((308, 224)), np.float32) / 255).permute(2, 0, 1)
                             for q in paths[k:k + 64]])
            h = model(pixel_values=((x - mean) / std).to(dev)).last_hidden_state
            e = torch.cat([F.normalize(h[:, 0], dim=1), F.normalize(h[:, 1:].mean(1), dim=1)], 1)  # CLS + mean patch
            E.append(F.normalize(e, dim=1).cpu().numpy())
    E = np.concatenate(E)
    np.save(cache, E)

n = len(tr_ids)
Et = torch.from_numpy(E).to(dev)
S_tr = (Et[:n] @ Et[:n].T).cpu().numpy(); np.fill_diagonal(S_tr, -1)
s_test = (Et[n:] @ Et[:n].T).max(1).values.cpu().numpy()  # test -> nearest train


def groups_at(tau):
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for a, b in zip(*np.where(np.triu(S_tr > tau, 1))):
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    return np.array([find(a) for a in range(n)])


te_res = Counter(size[q] for q in paths[n:])
res_of = [size[q] for q in paths[:n]]


def split(g, seed):
    """fill val group-by-group towards test resolution mix; resolutions absent in test never go to val"""
    rng = random.Random(seed)
    members = defaultdict(list)
    for a, gid in enumerate(g): members[gid].append(a)
    gl = list(members.values()); rng.shuffle(gl)
    target = {r: c / sum(te_res.values()) * args.val_frac * n for r, c in te_res.items()}
    got = Counter(); val = set()
    for m in gl:
        rc = Counter(res_of[a] for a in m)
        if any(r not in target for r in rc): continue
        if all(got[r] + c <= target[r] * 1.05 for r, c in rc.items()):
            val.update(m); got.update(rc)
    return np.array(sorted(val), dtype=int)


def ks(a, b):
    xs = np.sort(np.concatenate([a, b]))
    return np.abs(np.searchsorted(np.sort(a), xs, "right") / len(a) - np.searchsorted(np.sort(b), xs, "right") / len(b)).max()


rows, best = [], None
for tau in [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 1.01]:
    g = groups_at(tau)
    v = split(g, args.seed)
    isv = np.zeros(n, bool); isv[v] = True
    if isv.sum() < 50:  # one giant group -> nothing fits in val
        rows.append(dict(tau=tau, n_groups=len(set(g)), largest_group=Counter(g).most_common(1)[0][1], n_val=int(isv.sum()))); continue
    s_val = S_tr[np.ix_(np.where(isv)[0], np.where(~isv)[0])].max(1)
    r = dict(tau=tau, n_groups=len(set(g)), largest_group=Counter(g).most_common(1)[0][1], n_val=int(isv.sum()),
             val_med=np.median(s_val), test_med=np.median(s_test),
             val_p90=np.quantile(s_val, .9), test_p90=np.quantile(s_test, .9), KS=ks(s_val, s_test))
    rows.append(r)
    if r["n_val"] >= 0.12 * n and (best is None or r["KS"] < best[0]["KS"]):
        best = (r, g, isv)
print(pd.DataFrame(rows).round(3).to_string(index=False))

r, g, isv = best
print(f"\nchosen tau={r['tau']}  val={r['n_val']}  KS={r['KS']:.3f}")
man = pd.DataFrame({"image_id": tr_ids, "group": g, "split": np.where(isv, "val", "train"),
                    "res": [f"{w}x{h}" for w, h in res_of]})
man.to_csv(f"{args.out}/split_manifest.csv", index=False)
print(f"wrote {args.out}/split_manifest.csv")

In [ ]:
!python /content/make_group_split.py --root {DATA} --out {WORK}/split

## 3. Tile + zoom veri seti
Nesneler çok küçük: medyan kutu yaklaşık 30 px. Bu yüzden:
- **Tile:** 704 px'lik tile'lar (RF-DETR-L'nin çözünürlüğü), %25 overlap.
- **Zoom 1.33:** orijinalden 528 px'lik parça kesilip 704'e bicubic büyütülüyor. Küçük araçlar model için büyümüş oluyor.
- **Kesik kutular:** tile sınırında kesilen kutu, alanının en az %50'si görünüyorsa tutuluyor.
- **Boş tile'lar:** %15'i tutuluyor.
- **Tam görüntü:** her görüntü tam hâliyle de eğitime ekleniyor. Böylece model büyük araçları ve bağlamı da görüyor.
- **Tekrarlı kutular:** annotations.csv'deki birebir aynı kutular tekilleştiriliyor.
- `--val_in_train`: final model için val görüntüleri de train'e ekleniyor.

In [ ]:
%%writefile /content/preprocess.py
"""Build an RF-DETR (Roboflow COCO) dataset from the competition data. Originals are only read."""
import argparse, json, multiprocessing as mp, os, random, shutil, zlib
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image

CLASSES = ["car", "van", "truck", "bus"]

p = argparse.ArgumentParser()
p.add_argument("--root", default=".")
p.add_argument("--out", default="work/rfdetr_ds")
p.add_argument("--manifest", required=True)            # split_manifest.csv (scene-grouped split)
p.add_argument("--tile", type=int, default=704)
p.add_argument("--zoom", type=float, default=1.0)      # crop tile/zoom native px, upscale to tile
p.add_argument("--overlap", type=float, default=0.25)
p.add_argument("--min_vis", type=float, default=0.5)   # keep clipped box if >= this fraction visible
p.add_argument("--empty_keep", type=float, default=0.15)
p.add_argument("--val_in_train", action="store_true")  # final model: val also used for training
p.add_argument("--seed", type=int, default=42)
p.add_argument("--workers", type=int, default=os.cpu_count())
args = p.parse_args()

ann = pd.read_csv(f"{args.root}/train/annotations.csv")
ann = ann.drop_duplicates(["image_id", "x", "y", "w", "h"]).reset_index(drop=True)  # exact duplicate boxes
boxes = {k: v for k, v in ann.groupby("image_id")}

img_dir = f"{args.root}/train/images"
ids = sorted(f[:-4] for f in os.listdir(img_dir))
size = {i: Image.open(f"{img_dir}/{i}.jpg").size for i in ids}
man = pd.read_csv(args.manifest)
val = set(man.image_id[man.split == "val"]) & set(ids)
train = list(ids) if args.val_in_train else [i for i in ids if i not in val]
print(f"train {len(train)}  val {len(val)}")


def starts(L, T, S):
    if L <= T:
        return [0]
    n = int(np.ceil((L - T) / S)) + 1
    return [round(k * (L - T) / (n - 1)) for k in range(n)]


class Coco:
    def __init__(self, d):
        self.d = d
        os.makedirs(d, exist_ok=True)
        self.images, self.anns = [], []
        self.cats = [{"id": 0, "name": "vehicles", "supercategory": "none"}] + [
            {"id": k + 1, "name": c, "supercategory": "vehicles"} for k, c in enumerate(CLASSES)]

    def add(self, fname, W, H, bxs):
        iid = len(self.images)
        self.images.append({"id": iid, "file_name": fname, "width": W, "height": H})
        for x, y, w, h, lab in bxs:
            self.anns.append({"id": len(self.anns), "image_id": iid, "category_id": CLASSES.index(lab) + 1,
                              "bbox": [x, y, w, h], "area": w * h, "iscrowd": 0})

    def save(self):
        with open(f"{self.d}/_annotations.coco.json", "w") as f:
            json.dump({"images": self.images, "annotations": self.anns, "categories": self.cats}, f)
        print(f"{self.d}: {len(self.images)} images, {len(self.anns)} boxes")


def img_boxes(i):
    if i not in boxes:
        return np.zeros((0, 4)), []
    b = boxes[i]
    return b[["x", "y", "w", "h"]].values.astype(float), list(b.label)


def tile_image(i, d, keep_empty):
    r = random.Random(zlib.crc32(i.encode()) ^ args.seed)  # deterministic per image
    im = Image.open(f"{img_dir}/{i}.jpg").convert("RGB")
    W, H = im.size
    b, labs = img_boxes(i)
    T = round(args.tile / args.zoom)  # native crop size
    S, z = int(T * (1 - args.overlap)), args.tile / T
    out = []
    for y0 in starts(H, T, S):
        for x0 in starts(W, T, S):
            tw, th = min(T, W), min(T, H)
            x1 = np.clip(b[:, 0], x0, x0 + tw); y1 = np.clip(b[:, 1], y0, y0 + th)
            x2 = np.clip(b[:, 0] + b[:, 2], x0, x0 + tw); y2 = np.clip(b[:, 1] + b[:, 3], y0, y0 + th)
            cw, ch = np.clip(x2 - x1, 0, None), np.clip(y2 - y1, 0, None)
            vis = cw * ch / (b[:, 2] * b[:, 3]) if len(b) else np.zeros(0)
            bx = [(float(x1[k] - x0) * z, float(y1[k] - y0) * z, float(cw[k]) * z, float(ch[k]) * z, labs[k])
                  for k in np.where(vis >= args.min_vis)[0]]
            if not bx and r.random() > keep_empty:
                continue
            fname = f"{i}_{x0}_{y0}.jpg"
            crop = im.crop((x0, y0, x0 + tw, y0 + th))
            if z != 1:
                crop = crop.resize((round(tw * z), round(th * z)), Image.BICUBIC)
            crop.save(f"{d}/{fname}", quality=95)
            out.append((fname, crop.width, crop.height, bx))
    return out


def full_image(i, d):
    W, H = size[i]
    b, labs = img_boxes(i)
    shutil.copy2(f"{img_dir}/{i}.jpg", f"{d}/{i}.jpg")
    return [(f"{i}.jpg", W, H, [(*map(float, b[k]), labs[k]) for k in range(len(b))])]


def train_job(i):  # tiles + full image (model resizes it -> large vehicles + context)
    return tile_image(i, f"{args.out}/train", args.empty_keep) + full_image(i, f"{args.out}/train")


def val_job(i):
    return tile_image(i, f"{args.out}/valid", 0.15)


def build(coco, job, ids_):
    with ProcessPoolExecutor(args.workers, mp_context=mp.get_context("fork")) as ex:
        for entries in ex.map(job, ids_, chunksize=16):
            for e in entries:
                coco.add(*e)
    coco.save()


build(Coco(f"{args.out}/train"), train_job, train)
build(Coco(f"{args.out}/valid"), val_job, sorted(val))  # tiles, used only by RF-DETR's in-training eval
shutil.copytree(f"{args.out}/valid", f"{args.out}/test", dirs_exist_ok=True)  # rfdetr expects a test split
print("done")

## 4. Ortak araçlar

In [ ]:
import numpy as np, pandas as pd, torch, torchvision, time, pickle, re, math, copy, gc
from PIL import Image
from rfdetr import RFDETRLarge
pd.set_option('display.width', 250)
CLASSES = ['car', 'van', 'truck', 'bus']
dev = 'cuda'

# ---- tile'lı çıkarım -------------------------------------------------------------------------------
def load_model(ckpt):
    return RFDETRLarge.from_checkpoint(ckpt)

def starts(L, T, S):
    if L <= T: return [0]
    n = int(np.ceil((L - T) / S)) + 1
    return [round(k * (L - T) / (n - 1)) for k in range(n)]

def to_arr(d, x0, y0, z=1.0):
    names = np.array(d.data.get('class_name', []), dtype=object); keep = np.isin(names, CLASSES)
    b = d.xyxy[keep].astype(np.float32).copy() / z; b[:, [0, 2]] += x0; b[:, [1, 3]] += y0
    return b, d.confidence[keep].astype(np.float32), np.array([CLASSES.index(n) for n in names[keep]], np.int64)

@torch.no_grad()
def raw_predict(model, im, T_out=704, zoom=1.0, OV=0.25, thr=0.001, hflip=False):
    '''tile (zoom'lu) + tam görüntü tahminleri. tile kutularında iç kenara değenler işaretlenir (muhtemelen kesik).'''
    if hflip: im = im.transpose(Image.FLIP_LEFT_RIGHT)
    W, H = im.size; T = round(T_out / zoom); S = int(T * (1 - OV)); z = T_out / T
    crops, offs = [], []
    for y0 in starts(H, T, S):
        for x0 in starts(W, T, S):
            c = im.crop((x0, y0, x0 + min(T, W), y0 + min(T, H)))
            if z != 1: c = c.resize((round(c.width * z), round(c.height * z)), Image.BICUBIC)
            crops.append(c); offs.append((x0, y0, min(T, W), min(T, H)))
    dets = model.predict(crops + [im], threshold=thr)
    tb, ts, tl, te = [], [], [], []
    for d, (x0, y0, cw, ch) in zip(dets[:-1], offs):
        b, s, l = to_arr(d, x0, y0, z)
        e = np.zeros(len(b), bool)
        if x0 > 0: e |= b[:, 0] <= x0 + 2
        if y0 > 0: e |= b[:, 1] <= y0 + 2
        if x0 + cw < W: e |= b[:, 2] >= x0 + cw - 2
        if y0 + ch < H: e |= b[:, 3] >= y0 + ch - 2
        tb.append(b); ts.append(s); tl.append(l); te.append(e)
    fb, fs, fl = to_arr(dets[-1], 0, 0)
    r = dict(tile=(np.concatenate(tb), np.concatenate(ts), np.concatenate(tl), np.concatenate(te)), full=(fb, fs, fl))
    if hflip:
        for k in ('tile', 'full'):
            b = r[k][0]; b[:, [0, 2]] = W - b[:, [2, 0]]
    return r

# ---- birleştirme: kesik tile kutularını at + tam görüntüden sadece büyük kutular + sınıf bazlı NMS ----
TTA = {'z1.33': dict(zoom=4/3), 'z1.33_flip': dict(zoom=4/3, hflip=True), 'z1.6': dict(zoom=1.6)}
FMS, NMS_IOU, MAXDET = 176, 0.6, 500

def pass_preds(r, fms=FMS, topk=800):
    b, s, l, e = r['tile']; b, s, l = b[~e], s[~e], l[~e]
    fb, fs, fl = r['full']; k = np.maximum(fb[:, 2]-fb[:, 0], fb[:, 3]-fb[:, 1]) >= fms
    b, s, l = np.concatenate([b, fb[k]]), np.concatenate([s, fs[k]]), np.concatenate([l, fl[k]])
    o = np.argsort(-s)[:topk]; return b[o], s[o], l[o]

def cat(ps): return tuple(np.concatenate(x) for x in zip(*ps))

def do_nms(p, iou=NMS_IOU, max_det=MAXDET):
    b, s, l = p
    if not len(b): return p
    k = torchvision.ops.batched_nms(torch.from_numpy(b), torch.from_numpy(s), torch.from_numpy(l), iou).numpy()[:max_det]
    return b[k], s[k], l[k]

def fuse(passes_for_img): return do_nms(cat([pass_preds(r) for r in passes_for_img]))

def predict_tta(model, img_dir, ids, passes=TTA):
    RAW = {}
    for name, kw in passes.items():
        t0 = time.time()
        RAW[name] = {i: raw_predict(model, Image.open(f'{img_dir}/{i}.jpg').convert('RGB'), **kw) for i in ids}
        print(f'{name}: {(time.time()-t0)/60:.1f} dk', flush=True)
    return RAW

# ---- yarışma metriği: mAP@0.5, 4 sınıf ortalaması (all-point AP) ----------------------------------
def iou_xyxy(a, b):
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(2)
    return inter / ((a[:, 2:] - a[:, :2]).prod(1)[:, None] + (b[:, 2:] - b[:, :2]).prod(1)[None] - inter + 1e-9)

def ap_all_point(tp, conf, npos):
    if npos == 0: return np.nan
    o = np.argsort(-conf, kind='stable'); tp = tp[o]
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    rec, prec = ctp / npos, ctp / np.maximum(ctp + cfp, 1e-9)
    mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[0], prec, [0]])
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]).sum())

def evaluate(preds, ids, GT):
    res = {}
    for c in range(4):
        TP, CF, npos = [], [], 0
        for i in ids:
            gb, gl = GT.get(i, (np.zeros((0, 4)), np.zeros(0, int)))
            g = gb[gl == c]; g = np.c_[g[:, :2], g[:, :2] + g[:, 2:]]; npos += len(g)
            b, s, l = preds[i]; m = l == c; b, s = b[m], s[m]
            if not len(b): continue
            o = np.argsort(-s); b, s = b[o], s[o]; tp = np.zeros(len(b))
            if len(g):
                iou = iou_xyxy(b, g); used = np.zeros(len(g), bool)
                for k in range(len(b)):
                    j = np.argmax(np.where(used, -1, iou[k]))
                    if iou[k, j] >= 0.5 and not used[j]: tp[k] = 1; used[j] = True
            TP.append(tp); CF.append(s)
        res[CLASSES[c]] = ap_all_point(np.concatenate(TP), np.concatenate(CF), npos) if TP else 0.0
    res['mAP50'] = float(np.mean([res[c] for c in CLASSES]))
    return res

# ---- submission --------------------------------------------------------------------------------
def to_rows(preds, sizes, ids):
    out = []
    for i in ids:
        b, s, l = preds[i]; W, H = sizes[i]; parts = []
        for (x1, y1, x2, y2), sc, lb in zip(b, s, l):
            x1, y1 = int(round(max(0, min(W, x1)))), int(round(max(0, min(H, y1))))
            x2, y2 = int(round(max(0, min(W, x2)))), int(round(max(0, min(H, y2))))
            w, h = x2 - x1, y2 - y1
            if w <= 0 or h <= 0: continue
            parts.append(f'{CLASSES[lb]} {min(max(float(sc), 0.0), 1.0):.5f} {x1} {y1} {w} {h}')
        out.append((i, ' '.join(parts) if parts else 'none'))
    return out

def write_csv(rows, path): pd.DataFrame(rows, columns=['image_id', 'PredictionString']).to_csv(path, index=False)

def check_submission(path, sizes, sample_ids):
    lines = open(path, encoding='utf-8').read().splitlines()
    assert lines[0] == 'image_id,PredictionString' and len(lines) == len(sample_ids) + 1
    df = pd.read_csv(path, keep_default_na=False, dtype=str)
    assert df.image_id.tolist() == list(sample_ids), 'satır sırası sample ile aynı değil'
    pat = re.compile(r'^(?:none|(?:(?:car|van|truck|bus) (?:0|1)\.\d{5} \d+ \d+ [1-9]\d* [1-9]\d*)(?: (?:car|van|truck|bus) (?:0|1)\.\d{5} \d+ \d+ [1-9]\d* [1-9]\d*)*)$')
    bad = [i for i, ps in zip(df.image_id, df.PredictionString) if not pat.match(ps)]; assert not bad, bad[:3]
    for i, ps in zip(df.image_id, df.PredictionString):
        if ps == 'none': continue
        t = np.array(ps.split(' ')).reshape(-1, 6)[:, 2:].astype(int); W, H = sizes[i]
        assert (t[:, 0] + t[:, 2] <= W).all() and (t[:, 1] + t[:, 3] <= H).all(), i
    print(f'OK: {len(df)} satır, format/sıra/sınır doğru, {os.path.getsize(path)/1e6:.1f} MB')

# ---- val GT ------------------------------------------------------------------------------------
man = pd.read_csv(f'{WORK}/split/split_manifest.csv')
VAL_IDS = man.image_id[man.split == 'val'].tolist(); TR_IDS = man.image_id[man.split == 'train'].tolist()
ann = pd.read_csv(f'{DATA}/train/annotations.csv').drop_duplicates(['image_id', 'x', 'y', 'w', 'h'])
GT = {i: (g[['x','y','w','h']].values.astype(np.float32), g.label.map(CLASSES.index).values) for i, g in ann[ann.image_id.isin(VAL_IDS)].groupby('image_id')}
VSIZE = {i: Image.open(f'{DATA}/train/images/{i}.jpg').size for i in VAL_IDS}
print('train', len(TR_IDS), '| val', len(VAL_IDS))

## 5. Doğrulama deneyi (Run B): sahne bazlı split, zoom 1.33, augmentation, 6 epoch
Bu model yalnızca ayarları seçmek ve val skorunu ölçmek için eğitiliyor. Sınıflandırıcının α değeri de bu modelin val tahminleriyle ayarlandı.

Augmentation: yatay çevirme, parlaklık/kontrast, HSV, Affine ölçek 0.8–1.2.
Dikey çevirme ve 90° döndürme kullanılmadı, çünkü görüntüler eğik açılı drone çekimleri.
Overfitting 5. epoch civarında başladığı için 6 epoch yeterli.

In [ ]:
AUG = {
    "HorizontalFlip": {"p": 0.5},
    "RandomBrightnessContrast": {"brightness_limit": 0.2, "contrast_limit": 0.2, "p": 0.5},
    "HueSaturationValue": {"hue_shift_limit": 5, "sat_shift_limit": 20, "val_shift_limit": 10, "p": 0.3},
    "Affine": {"scale": (0.8, 1.2), "p": 0.3},
}
!python /content/preprocess.py --root {DATA} --out {WORK}/ds_zoom --tile 704 --zoom 1.3333 --manifest {WORK}/split/split_manifest.csv
OUT_B = f'{OUT_DIR}/out_B_zoom_aug'
RFDETRLarge().train(dataset_dir=f'{WORK}/ds_zoom', epochs=6, batch_size=42, grad_accum_steps=1, lr=1e-4,
                    output_dir=OUT_B, num_workers=16, progress_bar='tqdm', aug_config=AUG)

In [ ]:
# val: 3 geçişli TTA ham tahminleri + skorlar
modelB = load_model(f'{OUT_B}/checkpoint_best_total.pth')
PASSES = predict_tta(modelB, f'{DATA}/train/images', VAL_IDS)
pickle.dump(PASSES, open(f'{OUT_B}/val_raw_passes.pkl', 'wb'))
for name in TTA:
    print(f'{name:12s} tek geçiş:', round(evaluate({i: fuse([PASSES[name][i]]) for i in VAL_IDS}, VAL_IDS, GT)['mAP50'], 4))
VBEST = {i: fuse([PASSES[k][i] for k in TTA]) for i in VAL_IDS}
BASE = evaluate(VBEST, VAL_IDS, GT)
print('3 geçiş TTA:', {k: round(v, 4) for k, v in BASE.items()})   # beklenen ~0.8453

## 6. Kutu sınıflandırıcısı (ConvNeXt-Tiny)
Hata analizinde en büyük sorun **van→car karışıklığı** çıktı: GT van'ların yaklaşık %30'u car olarak tahmin ediliyordu. Van AP'si 0.74 civarındaydı.
Buna karşı ikinci aşama bir sınıflandırıcı eğitildi:
- **Veri:** train split'teki GT kutuları ve rastgele arka plan kutuları. Arka plan kutuları GT boyut dağılımından örnekleniyor ve hiçbir GT ile IoU'su 0.2'yi geçmiyor. 5 sınıf var: 4 araç + bg. Car sınıfı dengeleme için %60'a alt-örnekleniyor.
- **Girdi:** kutu merkezli, kenarı 2×max(w,h) olan kare bölge. GPU'da `roi_align` ile 160 px'e örnekleniyor, eğitimde kutunun 1.5 katı bölge 128 px'e alınıyor.
- **Augmentation (GPU içinde):** merkez ve ölçek jitter, en-boy oranı jitter, parlaklık/kontrast, 90° döndürme, çevirme. Kırpıntı düzeyinde yön bilgisi önemsiz olduğu için döndürme burada serbest.
- **Model:** `convnext_tiny.fb_in22k_ft_in1k`, 10 epoch, AdamW 3e-4, cosine, label smoothing 0.05.
- **Kullanım:** dedektörün kutuları ve etiketleri değişmiyor, sadece skor değişiyor: `score^(1-α) · p[etiket]^α`. Çıkarımda 8 görünüm TTA uygulanıyor (4 döndürme × çevirme).

In [ ]:
import timm, torch.nn.functional as F
from torchvision.ops import roi_align
from concurrent.futures import ThreadPoolExecutor
S0, CTX, IN, BG = 160, 2.0, 128, 4
C0 = S0 / 2; BOXPX = S0 / CTX; REG = BOXPX * 1.5      # saklanan crop'ta kutu 80 px, sınıflandırıcıya giden bölge 1.5x = 120 px
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1) * 255
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1) * 255

def region_crops(img, xyxy, size=S0, ctx=CTX):
    '''img: 3xHxW float GPU, xyxy: Nx4 -> N x 3 x size x size uint8, kutu merkezli kare (kenar = ctx*max(w,h))'''
    if not len(xyxy): return torch.zeros((0, 3, size, size), dtype=torch.uint8, device=dev)
    b = torch.as_tensor(xyxy, dtype=torch.float32, device=dev)
    c = (b[:, :2] + b[:, 2:]) / 2; half = (b[:, 2:] - b[:, :2]).amax(1).clamp(min=12) * ctx / 2
    rois = torch.cat([torch.zeros(len(b), 1, device=dev), c - half[:, None], c + half[:, None]], 1)
    out = torch.cat([roi_align(img[None], rois[k:k+4096], size, 1.0, 2, True) for k in range(0, len(rois), 4096)])
    return out.round().clamp(0, 255).to(torch.uint8)

def to_input(X, jitter=False):
    n = len(X); c = torch.full((n, 2), C0, device=dev); side = torch.full((n,), REG, device=dev); asp = torch.ones(n, device=dev)
    if jitter:
        c = c + (torch.rand(n, 2, device=dev) - .5) * 2 * .12 * BOXPX
        side = side * torch.empty(n, device=dev).uniform_(.85, 1.2); asp = torch.empty(n, device=dev).uniform_(.9, 1.1)
    hw, hh = side / 2 * asp, side / 2 / asp
    rois = torch.stack([torch.arange(n, device=dev, dtype=torch.float32), c[:, 0] - hw, c[:, 1] - hh, c[:, 0] + hw, c[:, 1] + hh], 1)
    x = roi_align(X.float(), rois, IN, 1.0, 2, True)
    if jitter:
        x = x * torch.empty(n, 1, 1, 1, device=dev).uniform_(.8, 1.2) + torch.empty(n, 1, 1, 1, device=dev).uniform_(-20, 20)
        x = torch.rot90(x, np.random.randint(4), (2, 3))
        if np.random.rand() < .5: x = x.flip(3)
    return ((x.clamp(0, 255) - MEAN) / STD).contiguous(memory_format=torch.channels_last)

@torch.no_grad()
def predict_probs(net, X, bs=2048, tta=True):   # tta: 4 döndürme x çevirme = 8 görünüm
    net.eval(); out = []
    for k in range(0, len(X), bs):
        x = to_input(X[k:k+bs]); acc = 0
        views = [torch.rot90(x, r, (2, 3)).flip(3) if f else torch.rot90(x, r, (2, 3)) for r in range(4) for f in (0, 1)] if tta else [x]
        with torch.autocast('cuda', torch.bfloat16):
            for v in views: acc = acc + net(v.contiguous(memory_format=torch.channels_last)).float().softmax(1)
        out.append(acc / len(views))
    return torch.cat(out)

@torch.no_grad()
def box_probs(net, preds, ids, img_dir, tta=True):
    out = {}
    with ThreadPoolExecutor(16) as ex:
        for i, arr in zip(ids, ex.map(lambda i: np.asarray(Image.open(f'{img_dir}/{i}.jpg').convert('RGB')), ids)):
            b = preds[i][0]
            if not len(b): out[i] = np.zeros((0, 5), np.float32); continue
            img = torch.from_numpy(arr.copy()).to(dev).permute(2, 0, 1).float()
            out[i] = predict_probs(net, region_crops(img, b), tta=tta).cpu().numpy()
    return out

def rescore(preds, probs, ids, alpha):
    R = {}
    for i in ids:
        b, s, l = preds[i]; q = probs[i][np.arange(len(l)), l]
        R[i] = (b, (s ** (1 - alpha) * np.clip(q, 1e-6, 1) ** alpha).astype(np.float32), l)
    return R

In [ ]:
# --- crop veri seti: GT kutuları + arka plan kutuları ---
rng = np.random.default_rng(0)
gt_all = ann[['x','y','w','h']].values.astype(np.float32)
ann_xyxy = {i: np.c_[g[['x','y']].values, g[['x','y']].values + g[['w','h']].values].astype(np.float32) for i, g in ann.groupby('image_id')}
ann_lab = {i: g.label.map(CLASSES.index).values for i, g in ann.groupby('image_id')}

def bg_boxes(i, W, H, n):
    '''GT boyut dağılımından örneklenmiş, hiçbir GT ile IoU>0.2 olmayan rastgele kutular'''
    g = ann_xyxy.get(i, np.zeros((0, 4), np.float32)); out = []
    for _ in range(n * 6):
        w, h = gt_all[rng.integers(len(gt_all)), 2:]
        x, y = rng.uniform(0, max(1, W - w)), rng.uniform(0, max(1, H - h)); bx = np.array([[x, y, x + w, y + h]], np.float32)
        if not len(g) or iou_xyxy(bx, g).max() < 0.2: out.append(bx[0])
        if len(out) == n: break
    return np.array(out, np.float32).reshape(-1, 4)

def build_crops(ids, n_bg, car_keep=1.0, seed=0):
    r = np.random.default_rng(seed); X, Y = [], []
    with ThreadPoolExecutor(16) as ex:
        for i, arr in zip(ids, ex.map(lambda i: np.asarray(Image.open(f'{DATA}/train/images/{i}.jpg').convert('RGB')), ids)):
            H, W = arr.shape[:2]; img = torch.from_numpy(arr).to(dev).permute(2, 0, 1).float()
            b, l = ann_xyxy.get(i, np.zeros((0, 4), np.float32)), ann_lab.get(i, np.zeros(0, int))
            k = (l != 0) | (r.random(len(l)) < car_keep); b, l = b[k], l[k]
            bb = bg_boxes(i, W, H, n_bg)
            X.append(region_crops(img, np.r_[b, bb])); Y.append(np.r_[l, np.full(len(bb), BG)])
    return torch.cat(X), torch.as_tensor(np.concatenate(Y), dtype=torch.long, device=dev)

Xtr, Ytr = build_crops(TR_IDS, n_bg=4, car_keep=0.6, seed=1)
Xva, Yva = build_crops(VAL_IDS, n_bg=4, car_keep=1.0, seed=2)
print(pd.DataFrame({'train': np.bincount(Ytr.cpu(), minlength=5), 'val': np.bincount(Yva.cpu(), minlength=5)}, index=CLASSES + ['bg']).T.to_string())

In [ ]:
# --- ConvNeXt-Tiny eğitimi ---
def val_report(net):
    pred = predict_probs(net, Xva, tta=False).argmax(1)
    cm = torch.zeros(5, 5, dtype=torch.long); cm.index_put_((Yva.cpu(), pred.cpu()), torch.ones(len(Yva), dtype=torch.long), accumulate=True)
    return (pred == Yva).float().mean().item(), pd.DataFrame((cm / cm.sum(1, keepdim=True) * 100).numpy().round(1), index=['GT ' + c for c in CLASSES + ['bg']], columns=CLASSES + ['bg'])

torch.manual_seed(0); np.random.seed(0)
clf = timm.create_model('convnext_tiny.fb_in22k_ft_in1k', pretrained=True, num_classes=5, drop_path_rate=0.1).to(dev).to(memory_format=torch.channels_last)
EPOCHS, BS, LR = 10, 256, 3e-4
opt = torch.optim.AdamW(clf.parameters(), lr=LR, weight_decay=0.05)
steps = EPOCHS * math.ceil(len(Xtr) / BS); warm = 300
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
for ep in range(EPOCHS):
    clf.train(); perm = torch.randperm(len(Xtr), device=dev); tl = 0
    for k in range(0, len(perm), BS):
        idx = perm[k:k+BS]
        with torch.autocast('cuda', torch.bfloat16): loss = F.cross_entropy(clf(to_input(Xtr[idx], jitter=True)), Ytr[idx], label_smoothing=0.05)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step(); tl += loss.item() * len(idx)
    acc, cmv = val_report(clf)
    print(f'ep {ep+1}  loss {tl/len(Xtr):.4f}  val acc {acc:.4f}  van->car {cmv.iloc[1,0]}%', flush=True)
print(cmv.to_string())
os.makedirs(f'{OUT_DIR}/classifier', exist_ok=True)
torch.save(clf.state_dict(), f'{OUT_DIR}/classifier/convnext_t_128_5cls.pth')
del Xtr, Ytr; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- α taraması: Run B TTA3 val kutuları üzerinde ---
VPROB = box_probs(clf, VBEST, VAL_IDS, f'{DATA}/train/images', tta=True)
rows = {'dedektör (TTA3)': BASE}
for a in [0.05, 0.08, 0.1, 0.12, 0.15, 0.2, 0.3, 0.5]:
    rows[f'α={a}'] = evaluate(rescore(VBEST, VPROB, VAL_IDS, a), VAL_IDS, GT)
print(pd.DataFrame(rows).T[CLASSES + ['mAP50']].round(4).to_string())   # α=0.12 -> ~0.8541
ALPHA = 0.12

## 7. Final: tüm veriyle eğitim → 6. epoch EMA → test TTA3 + sınıflandırıcı
Run B ile aynı ayarlar kullanılıyor, tek fark val görüntülerinin de eğitime girmesi (%23 daha fazla veri).
Model 7 epoch eğitiliyor ve her epoch checkpoint alınıyor. Run B'de en iyi EMA skoru son epoch'lara denk geldi. Val artık eğitimde olduğu için epoch seçimi ölçülemiyor, bu yüzden 6. epoch'un EMA ağırlıkları seçildi (public 0.80740).

In [ ]:
!python /content/preprocess.py --root {DATA} --out {WORK}/ds_full_zoom --tile 704 --zoom 1.3333 --manifest {WORK}/split/split_manifest.csv --val_in_train
OUT_F = f'{OUT_DIR}/out_FULL_zoom_aug'
RFDETRLarge().train(dataset_dir=f'{WORK}/ds_full_zoom', epochs=7, batch_size=42, grad_accum_steps=1, lr=1e-4,
                    output_dir=OUT_F, num_workers=16, progress_bar='tqdm', aug_config=AUG, checkpoint_interval=1)

In [ ]:
# 6. epoch (index 5) Lightning checkpoint'inden EMA ağırlıklarını çıkar, last_ema.pth formatında kaydet
ck = torch.load(f'{OUT_F}/checkpoint_5.ckpt', map_location='cpu', weights_only=False)
pe = torch.load(f'{OUT_F}/last_ema.pth', map_location='cpu', weights_only=False)
ema = ck['callbacks']['RFDETREMACallback']['average_model_state_dict']
ema_m = {k[len('module.model.'):]: v for k, v in ema.items() if k.startswith('module.model.')}
assert set(ema_m) == set(pe['model'])
new = copy.copy(pe); new['model'] = ema_m; new['epoch'] = 6; new['state_dict'] = {'model.' + k: v for k, v in ema_m.items()}
E6 = f'{OUT_F}/epoch6_ema.pth'; torch.save(new, E6)
del ck, pe, ema; gc.collect()

In [ ]:
# test: 3 geçişli TTA -> NMS -> sınıflandırıcı ile yeniden skorlama -> submission.csv
model6 = load_model(E6)
samp = pd.read_csv(f'{DATA}/sample_submission.csv', keep_default_na=False, dtype=str)
SUB_IDS = samp.image_id.tolist()
TE_SIZE = {i: Image.open(f'{DATA}/test/images/{i}.jpg').size for i in SUB_IDS}
RAW_TE = predict_tta(model6, f'{DATA}/test/images', SUB_IDS)
T6 = {i: fuse([RAW_TE[k][i] for k in TTA]) for i in SUB_IDS}

clf = timm.create_model('convnext_tiny.fb_in22k_ft_in1k', pretrained=False, num_classes=5).to(dev).to(memory_format=torch.channels_last)
clf.load_state_dict(torch.load(f'{OUT_DIR}/classifier/convnext_t_128_5cls.pth')); clf.eval()
TPROB = box_probs(clf, T6, SUB_IDS, f'{DATA}/test/images', tta=True)

SUB = f'{WORK}/submission.csv'
write_csv(to_rows(rescore(T6, TPROB, SUB_IDS, ALPHA), TE_SIZE, SUB_IDS), SUB)
check_submission(SUB, TE_SIZE, SUB_IDS)